In [23]:
#data structures
import pandas as pd
import numpy as np

#machine learning
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler

#metrics (performace and machine learning scores)
from sklearn.metrics import roc_auc_score
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import time #used for seeing how long it takes to run programs

np.random.seed(42)

In [24]:
train_val = pd.read_csv('data/train_values.csv') #The data values of each building
train_labels = pd.read_csv('data/train_labels.csv') #labels of data as damage severity ('damage_grade')

test_val = pd.read_csv('data/test_values.csv') #data values of contest submition (our AI would have to guess this, 
#but we won't know their 'damage_grade' for sure until DrivenData releases the answers.

In [37]:
#We combine the features and target label for data sampling and other transformations
df = pd.merge(train_val, train_labels, on='building_id')

In [38]:
df.head(3)

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
0,802906,6,487,12198,2,30,6,5,t,r,...,0,0,0,0,0,0,0,0,0,3
1,28830,8,900,2812,2,10,8,7,o,r,...,0,0,0,0,0,0,0,0,0,2
2,94947,21,363,8973,2,10,5,5,t,r,...,0,0,0,0,0,0,0,0,0,3


In [61]:
#Sampling the data so test runs don't take so long
dataSample = df.sample(frac=0.2)
#define X (data features) and y (target)
#X is all useful features
#y is target (final column 'default payment next month')

#Features to drop:
#building_id: for recordkeeping rather than data
#The target variable: damage_grade


features = dataSample.drop(['building_id', 'damage_grade'], axis = 1)

#Data adjustment
# encoder = OneHotEncoder(sparse_output=False, drop='first')

#These tables are in string type and must be converted
stringTables = ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']

one_hot_tables = pd.get_dummies(features[stringTables], prefix='education', dtype=int)


# scaler = MinMaxScaler()
# X = scaler.fit_transform(features)

#Now we replace our string tables with the encoded tables
X = features.drop(stringTables, axis = 1)
X = X.join(one_hot_tables)





y = dataSample['damage_grade']
X.head(4)

,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,has_superstructure_stone_flag,...,education_m,education_n,education_o,education_q,education_s,education_u,education_a,education_r,education_v,education_w
7576,6,376,9615,2,15,4,6,0,1,0,...,0,0,0,0,0,0,0,0,1,0
155188,10,735,519,2,40,7,4,1,1,0,...,0,0,0,0,0,0,0,0,1,0
45749,0,62,8166,2,50,8,7,1,1,0,...,0,0,0,0,0,0,0,0,1,0
22039,26,619,11351,1,20,6,3,0,1,0,...,0,0,0,0,0,0,0,0,1,0


In [64]:
#This function creates a K-Nearest-Neighbors classifier
#X: features
#y: target variable
#randomState: the random seed
#testSize: ratio of test to training data
#crossValNum: The number of cross validation groups
#nNeighbors: the number of Neighbors used in calculations
def testKNearestNeighbours(X, y, randomState, testSize, crossValNum, nNeighbors):
    start_time = time.perf_counter()
    KNN = KNeighborsClassifier(n_neighbors = nNeighbors)
    #Separate training and testing data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=testSize, random_state=randomState, stratify=y)
    #train the model
    KNN.fit(X_train, y_train)
    #cross validation at k = 4 and k = 5
    cross_val_scores = cross_val_score(KNN, X, y, cv=crossValNum)
    #Print results
    score = KNN.score(X_test, y_test)

    end_time = time.perf_counter()
    # predictions = classifier.predict(xTest)
    print('K-Nearest-Neighbours classifier with test size of ', testSize * 100, '%, ', crossValNum, ' cross validation groups, and ', nNeighbors, ' neighbors')
    print('Accuracy: ', score)
    print('Cross validation Accuracy', cross_val_scores)
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.4f} seconds")

    #This specific code I adapted from https://proclusacademy.com/blog/practical/precision-recall-f1-score-sklearn/
    y_test_predictions = KNN.predict(X_test)
    precision = precision_score(y_test, y_test_predictions, average='weighted')
    recall = recall_score(y_test, y_test_predictions, average='weighted')
    f1score = f1_score(y_test, y_test_predictions, average='weighted')
    print(f"Precision = {precision}")
    print(f"Recall = {recall}")
    print(f"F1 Score = {f1score}")
    #end of copied code
    #This stuff is for graphing
    # auc_score = roc_auc_score(y_test, KNN.predict_proba(X_test)[:, 1], multi_class='ovr')
    auc_score = roc_auc_score(y_test, KNN.predict_proba(X_test), multi_class='ovr')
    print(f"ROC AUC Score: {auc_score}")
    
    metrics = np.array([nNeighbors, score, precision, recall, f1score, elapsed_time, auc_score])
    cvMetrics = np.array(nNeighbors)
    cvMetrics = np.append(cvMetrics, np.array(cross_val_scores))

    # print(cvMetrics)
    return metrics, cvMetrics

In [65]:
testKNearestNeighbours(X, y, 123, 0.2, 4, 3)

K-Nearest-Neighbours classifier with test size of  20.0 %,  4  cross validation groups, and  3  neighbors
Accuracy:  0.6438986953184958
Cross validation Accuracy [0.64228703 0.64036838 0.64082886 0.64750576]
Elapsed time: 15.6418 seconds
Precision = 0.6448521376818722
Recall = 0.6438986953184958
F1 Score = 0.6441014932179164
ROC AUC Score: 0.7423424115457561


(array([ 3.        ,  0.6438987 ,  0.64485214,  0.6438987 ,  0.64410149,
        15.6417593 ,  0.74234241]),
 array([3.        , 0.64228703, 0.64036838, 0.64082886, 0.64750576]))